## Product Text Dataset and NLP Problem Definition

This section prepares product-level text data from the cleaned e-commerce transactions and defines how Transformer-based NLP will be used in CommerceIQ for semantic product understanding and retrieval.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/clean_transactions.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDate range:")
print(df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

Dataset shape: (536642, 18)

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'TransactionType', 'IsCancellation', 'IsNegativeQuantity', 'IsZeroPrice']

Date range:
2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [2]:
product_text_df = df[
    [
        "StockCode",
        "Description",
        "Country",
        "TransactionType",
        "Quantity",
        "Revenue"
    ]
].copy()

print("Shape:", product_text_df.shape)

print("\nMissing Description:")
print(product_text_df["Description"].isna().sum())

print("\nUnique StockCodes:")
print(product_text_df["StockCode"].nunique())

print("\nUnique Descriptions:")
print(product_text_df["Description"].nunique())

Shape: (536642, 6)

Missing Description:
1454

Unique StockCodes:
4070

Unique Descriptions:
4223


In [3]:
product_catalog = (
    df[df["Description"].notna()]
    .groupby("StockCode")
    .agg(
        Product_Name=("Description", "first"),
        Total_Units_Sold=("Quantity", lambda x: x[x > 0].sum()),
        Total_Revenue=("Revenue", "sum"),
        Order_Count=("Invoice", "nunique"),
        Customer_Count=("Customer ID", "nunique")
    )
    .reset_index()
)

print("Product catalog shape:", product_catalog.shape)

print("\nFirst 10 products:")
print(product_catalog.head(10))

Product catalog shape: (3958, 6)

First 10 products:
  StockCode                  Product_Name  Total_Units_Sold  Total_Revenue  \
0     10002   INFLATABLE POLITICAL GLOBE                860         759.89   
1     10080      GROOVY CACTUS INFLATABLE               325         119.09   
2     10120                  DOGGY RUBBER               192          40.32   
3    10123C         HEARTS WRAPPING TAPE                  5           3.25   
4    10124A   SPOTS ON RED BOOKCOVER TAPE                16           6.72   
5    10124G      ARMY CAMO BOOKCOVER TAPE                17           7.14   
6     10125       MINI FUNKY DESIGN TAPES              1295         993.99   
7     10133  COLOURING PENCILS BROWN TUBE              2856        1535.40   
8     10135  COLOURING PENCILS BROWN TUBE              2229        2203.64   
9     11001   ASSTD DESIGN RACING CAR PEN              1615        2152.39   

   Order_Count  Customer_Count  
0           71              40  
1           23        

In [4]:
product_catalog["Product_Name"] = (
    product_catalog["Product_Name"]
    .astype(str)
    .str.strip()
)

text_quality = pd.DataFrame({
    "Metric": [
        "Total products",
        "Missing product names",
        "Empty product names",
        "Unique product names",
        "Duplicate product names"
    ],
    "Value": [
        len(product_catalog),
        product_catalog["Product_Name"].isna().sum(),
        (product_catalog["Product_Name"] == "").sum(),
        product_catalog["Product_Name"].nunique(),
        product_catalog["Product_Name"].duplicated().sum()
    ]
})

text_quality

,Metric,Value
0,Total products,3958
1,Missing product names,0
2,Empty product names,0
3,Unique product names,3808
4,Duplicate product names,150


In [5]:
print("Sample product descriptions:\n")

for i, text in enumerate(product_catalog["Product_Name"].head(30), start=1):
    print(f"{i}. {text}")

Sample product descriptions:

1. INFLATABLE POLITICAL GLOBE
2. GROOVY CACTUS INFLATABLE
3. DOGGY RUBBER
4. HEARTS WRAPPING TAPE
5. SPOTS ON RED BOOKCOVER TAPE
6. ARMY CAMO BOOKCOVER TAPE
7. MINI FUNKY DESIGN TAPES
8. COLOURING PENCILS BROWN TUBE
9. COLOURING PENCILS BROWN TUBE
10. ASSTD DESIGN RACING CAR PEN
11. FAN BLACK FRAME
12. PAPER POCKET TRAVELING FAN
13. ASSORTED COLOURS SILK FAN
14. SANDALWOOD FAN
15. PINK PAPER PARASOL
16. BLUE PAPER PARASOL
17. PURPLE PAPER PARASOL
18. RED PAPER PARASOL
19. EDWARDIAN PARASOL BLACK
20. EDWARDIAN PARASOL NATURAL
21. EDWARDIAN PARASOL PINK
22. EDWARDIAN PARASOL BLACK
23. EDWARDIAN PARASOL NATURAL
24. EDWARDIAN PARASOL PINK
25. BLUE POLKADOT GARDEN PARASOL
26. PINK POLKADOT GARDEN PARASOL
27. ICE CREAM DESIGN GARDEN PARASOL
28. FAIRY CAKE DESIGN UMBRELLA
29. FAIRY CAKE DESIGN UMBRELLA
30. SMALL FOLDING SCISSOR(POINTED EDGE)


In [6]:
product_catalog["Text_Length"] = (
    product_catalog["Product_Name"]
    .str.len()
)

product_catalog["Word_Count"] = (
    product_catalog["Product_Name"]
    .str.split()
    .str.len()
)

print("Text length statistics:")
print(product_catalog["Text_Length"].describe())

print("\nWord count statistics:")
print(product_catalog["Word_Count"].describe())

print("\nLongest product descriptions:")
print(
    product_catalog[
        ["StockCode", "Product_Name", "Text_Length", "Word_Count"]
    ]
    .sort_values("Text_Length", ascending=False)
    .head(10)
)

Text length statistics:
count    3958.000000
mean       26.853967
std         5.437345
min         1.000000
25%        23.000000
50%        28.000000
75%        31.000000
max        35.000000
Name: Text_Length, dtype: float64

Word count statistics:
count    3958.000000
mean        4.367105
std         1.069436
min         1.000000
25%         4.000000
50%         4.000000
75%         5.000000
max         8.000000
Name: Word_Count, dtype: float64

Longest product descriptions:
     StockCode                         Product_Name  Text_Length  Word_Count
680      21615  4 LAVENDER BOTANICAL DINNER CANDLES           35           5
677      21609  SET 12 LAVENDER  BOTANICAL T-LIGHTS           35           5
704      21640  ASSORTED TUTTI FRUTTI  FOB NOTEBOOK           35           5
3847    90184B  AMETHYST CHUNKY BEAD BRACELET W STR           35           6
3843    90183B  AMETHYST DROP EARRINGS W LONG BEADS           35           6
1812     22944  CHRISTMAS METAL POSTCARD WITH BELLS     

In [8]:
duplicate_descriptions = (
    product_catalog[
        product_catalog["Product_Name"].duplicated(keep=False)
    ]
    .sort_values("Product_Name")
)

print("Products sharing identical descriptions:",
    len(duplicate_descriptions))

print("\nExamples:")
print(duplicate_descriptions.head(20))

Products sharing identical descriptions: 282

Examples:
     StockCode                       Product_Name  Total_Units_Sold  \
3351    85034a    3 GARDENIA MORRIS BOXED CANDLES                 3   
3348    85034A    3 GARDENIA MORRIS BOXED CANDLES               522   
3349    85034B  3 WHITE CHOC MORRIS BOXED CANDLES               945   
3352    85034b  3 WHITE CHOC MORRIS BOXED CANDLES                 1   
3030    84558A       3D DOG PICTURE PLAYING CARDS               333   
3031    84558a       3D DOG PICTURE PLAYING CARDS                 5   
3033    84559B           3D SHEET OF CAT STICKERS               237   
3035    84559b           3D SHEET OF CAT STICKERS                23   
3032    84559A           3D SHEET OF DOG STICKERS               397   
3034    84559a           3D SHEET OF DOG STICKERS                29   
2792    72801C         4 ROSE PINK DINNER CANDLES               145   
2795    72801c         4 ROSE PINK DINNER CANDLES                39   
2793    72801D       

In [9]:
nlp_use_cases = pd.DataFrame({
    "Use Case": [
        "Semantic Product Search",
        "Similar Product Retrieval",
        "Product Description Understanding",
        "Natural Language Product Queries"
    ],
    "Purpose": [
        "Find products based on meaning rather than exact keywords",
        "Retrieve products with semantically similar descriptions",
        "Convert product text into meaningful numerical representations",
        "Support future RAG and AI-agent queries over the product catalog"
    ]
})

nlp_use_cases

,Use Case,Purpose
0,Semantic Product Search,Find products based on meaning rather than exa...
1,Similar Product Retrieval,Retrieve products with semantically similar de...
2,Product Description Understanding,Convert product text into meaningful numerical...
3,Natural Language Product Queries,Support future RAG and AI-agent queries over t...


In [10]:
product_catalog.to_csv(
    "../data/product_nlp_catalog.csv",
    index=False
)

print("Saved product NLP catalog.")
print("Path: ../data/product_nlp_catalog.csv")
print("Shape:", product_catalog.shape)

Saved product NLP catalog.
Path: ../data/product_nlp_catalog.csv
Shape: (3958, 8)


In [11]:
print("FINAL VALIDATION")
print("-" * 40)

print("Rows:", len(product_catalog))
print("Columns:", len(product_catalog.columns))
print("Missing Product_Name:", product_catalog["Product_Name"].isna().sum())
print("Empty Product_Name:",
    (product_catalog["Product_Name"] == "").sum())
print("Unique products:", product_catalog["StockCode"].nunique())
print("Unique descriptions:", product_catalog["Product_Name"].nunique())

FINAL VALIDATION
----------------------------------------
Rows: 3958
Columns: 8
Missing Product_Name: 0
Empty Product_Name: 0
Unique products: 3958
Unique descriptions: 3808


## Product Text Cleaning and Normalization

This section normalizes product descriptions while preserving information that may be important for product semantics, including numbers, colors, sizes, quantities, and product attributes. The original descriptions are retained for comparison and downstream use.

In [12]:
print("Raw product descriptions:")
print(
    product_catalog[
        ["StockCode", "Product_Name"]
    ].head(20).to_string(index=False)
)

Raw product descriptions:
StockCode                 Product_Name
    10002   INFLATABLE POLITICAL GLOBE
    10080     GROOVY CACTUS INFLATABLE
    10120                 DOGGY RUBBER
   10123C         HEARTS WRAPPING TAPE
   10124A  SPOTS ON RED BOOKCOVER TAPE
   10124G     ARMY CAMO BOOKCOVER TAPE
    10125      MINI FUNKY DESIGN TAPES
    10133 COLOURING PENCILS BROWN TUBE
    10135 COLOURING PENCILS BROWN TUBE
    11001  ASSTD DESIGN RACING CAR PEN
    15030              FAN BLACK FRAME
    15034   PAPER POCKET TRAVELING FAN
    15036    ASSORTED COLOURS SILK FAN
    15039               SANDALWOOD FAN
   15044A           PINK PAPER PARASOL
   15044B           BLUE PAPER PARASOL
   15044C         PURPLE PAPER PARASOL
   15044D            RED PAPER PARASOL
  15056BL      EDWARDIAN PARASOL BLACK
   15056N    EDWARDIAN PARASOL NATURAL


In [13]:
import re

def normalize_product_text(text):
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Replace common separators with spaces
    text = re.sub(r"[/&,-]+", " ", text)

    # Keep letters and numbers
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


product_catalog["Normalized_Text"] = (
    product_catalog["Product_Name"]
    .apply(normalize_product_text)
)

print(
    product_catalog[
        ["Product_Name", "Normalized_Text"]
    ].head(20).to_string(index=False)
)

                Product_Name              Normalized_Text
  INFLATABLE POLITICAL GLOBE   inflatable political globe
    GROOVY CACTUS INFLATABLE     groovy cactus inflatable
                DOGGY RUBBER                 doggy rubber
        HEARTS WRAPPING TAPE         hearts wrapping tape
 SPOTS ON RED BOOKCOVER TAPE  spots on red bookcover tape
    ARMY CAMO BOOKCOVER TAPE     army camo bookcover tape
     MINI FUNKY DESIGN TAPES      mini funky design tapes
COLOURING PENCILS BROWN TUBE colouring pencils brown tube
COLOURING PENCILS BROWN TUBE colouring pencils brown tube
 ASSTD DESIGN RACING CAR PEN  asstd design racing car pen
             FAN BLACK FRAME              fan black frame
  PAPER POCKET TRAVELING FAN   paper pocket traveling fan
   ASSORTED COLOURS SILK FAN    assorted colours silk fan
              SANDALWOOD FAN               sandalwood fan
          PINK PAPER PARASOL           pink paper parasol
          BLUE PAPER PARASOL           blue paper parasol
        PURPLE

In [14]:
comparison = product_catalog[
    ["StockCode", "Product_Name", "Normalized_Text"]
].copy()

comparison["Changed"] = (
    comparison["Product_Name"] != comparison["Normalized_Text"]
)

print("Descriptions changed:", comparison["Changed"].sum())
print(
    "Descriptions unchanged:",
    (~comparison["Changed"]).sum()
)

print("\nExamples of normalization:")
print(
    comparison[comparison["Changed"]]
    .head(20)
    .to_string(index=False)
)

Descriptions changed: 3941
Descriptions unchanged: 17

Examples of normalization:
StockCode                 Product_Name              Normalized_Text  Changed
    10002   INFLATABLE POLITICAL GLOBE   inflatable political globe     True
    10080     GROOVY CACTUS INFLATABLE     groovy cactus inflatable     True
    10120                 DOGGY RUBBER                 doggy rubber     True
   10123C         HEARTS WRAPPING TAPE         hearts wrapping tape     True
   10124A  SPOTS ON RED BOOKCOVER TAPE  spots on red bookcover tape     True
   10124G     ARMY CAMO BOOKCOVER TAPE     army camo bookcover tape     True
    10125      MINI FUNKY DESIGN TAPES      mini funky design tapes     True
    10133 COLOURING PENCILS BROWN TUBE colouring pencils brown tube     True
    10135 COLOURING PENCILS BROWN TUBE colouring pencils brown tube     True
    11001  ASSTD DESIGN RACING CAR PEN  asstd design racing car pen     True
    15030              FAN BLACK FRAME              fan black frame    

In [15]:
empty_normalized = (
    product_catalog["Normalized_Text"]
    .isna()
    | (product_catalog["Normalized_Text"] == "")
)

print("Empty normalized descriptions:", empty_normalized.sum())

if empty_normalized.sum() > 0:
    print("\nAffected products:")
    print(
        product_catalog.loc[
            empty_normalized,
            ["StockCode", "Product_Name"]
        ]
    )

Empty normalized descriptions: 1

Affected products:
    StockCode Product_Name
481     21275            ?


In [16]:
product_catalog["Normalized_Text_Length"] = (
    product_catalog["Normalized_Text"].str.len()
)

product_catalog["Normalized_Word_Count"] = (
    product_catalog["Normalized_Text"]
    .str.split()
    .str.len()
)

print("Normalized text length:")
print(product_catalog["Normalized_Text_Length"].describe())

print("\nNormalized word count:")
print(product_catalog["Normalized_Word_Count"].describe())

Normalized text length:
count    3958.000000
mean       26.746589
std         5.442623
min         0.000000
25%        23.000000
50%        27.000000
75%        31.000000
max        35.000000
Name: Normalized_Text_Length, dtype: float64

Normalized word count:
count    3958.000000
mean        4.467408
std         1.114757
min         0.000000
25%         4.000000
50%         4.000000
75%         5.000000
max         8.000000
Name: Normalized_Word_Count, dtype: float64


In [17]:
duplicate_normalized = (
    product_catalog[
        product_catalog["Normalized_Text"].duplicated(keep=False)
    ]
    .sort_values("Normalized_Text")
)

print(
    "Products sharing normalized descriptions:",
    len(duplicate_normalized)
)

print(
    "Unique normalized descriptions:",
    product_catalog["Normalized_Text"].nunique()
)

print("\nExamples:")
print(
    duplicate_normalized[
        ["StockCode", "Product_Name", "Normalized_Text"]
    ]
    .head(30)
    .to_string(index=False)
)

Products sharing normalized descriptions: 292
Unique normalized descriptions: 3802

Examples:
StockCode                       Product_Name                    Normalized_Text
   85034A    3 GARDENIA MORRIS BOXED CANDLES    3 gardenia morris boxed candles
   85034a    3 GARDENIA MORRIS BOXED CANDLES    3 gardenia morris boxed candles
   85034B  3 WHITE CHOC MORRIS BOXED CANDLES  3 white choc morris boxed candles
   85034b  3 WHITE CHOC MORRIS BOXED CANDLES  3 white choc morris boxed candles
   84558a       3D DOG PICTURE PLAYING CARDS       3d dog picture playing cards
   84558A       3D DOG PICTURE PLAYING CARDS       3d dog picture playing cards
   84559B           3D SHEET OF CAT STICKERS           3d sheet of cat stickers
   84559b           3D SHEET OF CAT STICKERS           3d sheet of cat stickers
   84559A           3D SHEET OF DOG STICKERS           3d sheet of dog stickers
   84559a           3D SHEET OF DOG STICKERS           3d sheet of dog stickers
   72801C         4 ROSE P

In [18]:
sample_check = product_catalog[
    product_catalog["Product_Name"].str.contains(
        r"\d|red|white|pink|blue|green",
        case=False,
        na=False
    )
][
    ["StockCode", "Product_Name", "Normalized_Text"]
].head(20)

print(sample_check.to_string(index=False))

StockCode                        Product_Name                     Normalized_Text
   10124A         SPOTS ON RED BOOKCOVER TAPE         spots on red bookcover tape
   15044A                  PINK PAPER PARASOL                  pink paper parasol
   15044B                  BLUE PAPER PARASOL                  blue paper parasol
   15044D                   RED PAPER PARASOL                   red paper parasol
   15056P              EDWARDIAN PARASOL PINK              edwardian parasol pink
   15056p              EDWARDIAN PARASOL PINK              edwardian parasol pink
   15058A        BLUE POLKADOT GARDEN PARASOL        blue polkadot garden parasol
   15058B        PINK POLKADOT GARDEN PARASOL        pink polkadot garden parasol
   16151A     FLOWERS HANDBAG blue and orange     flowers handbag blue and orange
   16156S               WRAP PINK FAIRY CAKES               wrap pink fairy cakes
   16161M                    WRAP  PINK FLOCK                     wrap pink flock
   16169E       

In [19]:
product_catalog.to_csv(
    "../data/product_nlp_catalog.csv",
    index=False
)

print("NLP-ready product catalog saved.")
print("Shape:", product_catalog.shape)
print("Path: ../data/product_nlp_catalog.csv")

NLP-ready product catalog saved.
Shape: (3958, 11)
Path: ../data/product_nlp_catalog.csv


In [20]:
print("FINAL VALIDATION")
print("-" * 40)

print("Products:", len(product_catalog))
print(
    "Missing original descriptions:",
    product_catalog["Product_Name"].isna().sum()
)
print(
    "Missing normalized descriptions:",
    product_catalog["Normalized_Text"].isna().sum()
)
print(
    "Empty normalized descriptions:",
    (product_catalog["Normalized_Text"] == "").sum()
)
print(
    "Unique normalized descriptions:",
    product_catalog["Normalized_Text"].nunique()
)
print(
    "Columns:",
    product_catalog.columns.tolist()
)

FINAL VALIDATION
----------------------------------------
Products: 3958
Missing original descriptions: 0
Missing normalized descriptions: 0
Empty normalized descriptions: 1
Unique normalized descriptions: 3802
Columns: ['StockCode', 'Product_Name', 'Total_Units_Sold', 'Total_Revenue', 'Order_Count', 'Customer_Count', 'Text_Length', 'Word_Count', 'Normalized_Text', 'Normalized_Text_Length', 'Normalized_Word_Count']


In [21]:
# Remove products with no meaningful textual description
product_catalog = product_catalog[
    product_catalog["Normalized_Text"].str.strip() != ""
].copy()

print("Products remaining:", len(product_catalog))
print(
    "Empty normalized descriptions:",
    (product_catalog["Normalized_Text"].str.strip() == "").sum()
)

print(
    "Unique normalized descriptions:",
    product_catalog["Normalized_Text"].nunique()
)

Products remaining: 3957
Empty normalized descriptions: 0
Unique normalized descriptions: 3801


In [22]:
product_catalog.to_csv(
    "../data/product_nlp_catalog.csv",
    index=False
)

print("Updated NLP catalog saved.")
print("Final shape:", product_catalog.shape)

Updated NLP catalog saved.
Final shape: (3957, 11)


In [23]:
print("FINAL_VALIDATION")
print("-" * 40)

print("Products:", len(product_catalog))
print("Missing Product_Name:", product_catalog["Product_Name"].isna().sum())
print("Empty Normalized_Text:",
    (product_catalog["Normalized_Text"].str.strip() == "").sum())
print("Unique StockCodes:", product_catalog["StockCode"].nunique())
print("Unique descriptions:", product_catalog["Product_Name"].nunique())
print("Unique normalized descriptions:",
    product_catalog["Normalized_Text"].nunique())

FINAL_VALIDATION
----------------------------------------
Products: 3957
Missing Product_Name: 0
Empty Normalized_Text: 0
Unique StockCodes: 3957
Unique descriptions: 3807
Unique normalized descriptions: 3801


## Tokenization and Transformer Inputs

This section converts normalized product descriptions into tokenized Transformer inputs. We examine subword tokenization, token IDs, attention masks, and sequence lengths before generating Transformer representations.

In [26]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("Model maximum length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

e:\commerceiq\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sadiya Sajid\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer: distilbert-base-uncased
Vocabulary size: 30522
Model maximum length: 512


In [27]:
import pandas as pd
import numpy as np

product_catalog = pd.read_csv(
    "../data/product_nlp_catalog.csv"
)

print("Product catalog shape:", product_catalog.shape)

print("\nColumns:")
print(product_catalog.columns.tolist())

print("\nSample:")
print(
    product_catalog[
        ["StockCode", "Product_Name", "Normalized_Text"]
    ].head(10)
)

Product catalog shape: (3957, 11)

Columns:
['StockCode', 'Product_Name', 'Total_Units_Sold', 'Total_Revenue', 'Order_Count', 'Customer_Count', 'Text_Length', 'Word_Count', 'Normalized_Text', 'Normalized_Text_Length', 'Normalized_Word_Count']

Sample:
  StockCode                  Product_Name               Normalized_Text
0     10002    INFLATABLE POLITICAL GLOBE    inflatable political globe
1     10080      GROOVY CACTUS INFLATABLE      groovy cactus inflatable
2     10120                  DOGGY RUBBER                  doggy rubber
3    10123C          HEARTS WRAPPING TAPE          hearts wrapping tape
4    10124A   SPOTS ON RED BOOKCOVER TAPE   spots on red bookcover tape
5    10124G      ARMY CAMO BOOKCOVER TAPE      army camo bookcover tape
6     10125       MINI FUNKY DESIGN TAPES       mini funky design tapes
7     10133  COLOURING PENCILS BROWN TUBE  colouring pencils brown tube
8     10135  COLOURING PENCILS BROWN TUBE  colouring pencils brown tube
9     11001   ASSTD DESIGN R

In [28]:
sample_text = product_catalog["Normalized_Text"].iloc[0]

tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original text:")
print(sample_text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

print("\nNumber of tokens:", len(tokens))

Original text:
inflatable political globe

Tokens:
['in', '##fl', '##atable', 'political', 'globe']

Token IDs:
[1999, 10258, 27892, 2576, 7595]

Number of tokens: 5


In [29]:
encoded_sample = tokenizer(
    sample_text,
    add_special_tokens=True
)

print("Input IDs:")
print(encoded_sample["input_ids"])

print("\nAttention Mask:")
print(encoded_sample["attention_mask"])

print("\nDecoded text:")
print(
    tokenizer.decode(
        encoded_sample["input_ids"]
    )
)

Input IDs:
[101, 1999, 10258, 27892, 2576, 7595, 102]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1]

Decoded text:
[CLS] inflatable political globe [SEP]


In [30]:
print("CLS token:", tokenizer.cls_token)
print("CLS token ID:", tokenizer.cls_token_id)

print("SEP token:", tokenizer.sep_token)
print("SEP token ID:", tokenizer.sep_token_id)

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

print("UNK token:", tokenizer.unk_token)
print("UNK token ID:", tokenizer.unk_token_id)

CLS token: [CLS]
CLS token ID: 101
SEP token: [SEP]
SEP token ID: 102
PAD token: [PAD]
PAD token ID: 0
UNK token: [UNK]
UNK token ID: 100


In [31]:
tokenized_products = tokenizer(
    product_catalog["Normalized_Text"].tolist(),
    padding=False,
    truncation=True,
    add_special_tokens=True
)

print("Number of tokenized products:",
    len(tokenized_products["input_ids"]))

print("First tokenized product:")
print(tokenized_products["input_ids"][0])

Number of tokenized products: 3957
First tokenized product:
[101, 1999, 10258, 27892, 2576, 7595, 102]


In [32]:
token_lengths = np.array([
    len(ids)
    for ids in tokenized_products["input_ids"]
])

print("Token length statistics:")
print(pd.Series(token_lengths).describe())

print("\nMaximum token length:", token_lengths.max())
print("Minimum token length:", token_lengths.min())
print("Median token length:", np.median(token_lengths))

Token length statistics:
count    3957.000000
mean        7.318170
std         1.504495
min         3.000000
25%         6.000000
50%         7.000000
75%         8.000000
max        14.000000
dtype: float64

Maximum token length: 14
Minimum token length: 3
Median token length: 7.0


In [33]:
max_length = tokenizer.model_max_length

truncated_count = np.sum(
    token_lengths > max_length
)

print("Tokenizer maximum length:", max_length)
print("Products exceeding maximum length:", truncated_count)
print(
    "Percentage exceeding maximum:",
    round(truncated_count / len(token_lengths) * 100, 4),
    "%"
)

Tokenizer maximum length: 512
Products exceeding maximum length: 0
Percentage exceeding maximum: 0.0 %


In [34]:
token_length_df = product_catalog[
    ["StockCode", "Product_Name", "Normalized_Text"]
].copy()

token_length_df["Token_Length"] = token_lengths

print(
    token_length_df
    .sort_values("Token_Length", ascending=False)
    .head(15)
    .to_string(index=False)
)

   StockCode                        Product_Name                     Normalized_Text  Token_Length
       23650  SET 10 CARDS FILIGREE BAUBLE 16961  set 10 cards filigree bauble 16961            14
      90036F FLOWER GLASS GARLD NECKL36"TURQUOIS flower glass garld neckl36 turquois            14
       23239   SET OF 4 KNICK KNACK TINS POPPIES   set of 4 knick knack tins poppies            13
gift_0001_40  Dotcomgiftshop Gift Voucher £40.00   dotcomgiftshop gift voucher 40 00            13
      90036E FLOWER GLASS GARLD NECKL36"AMETHYST flower glass garld neckl36 amethyst            13
       23240    SET OF 4 KNICK KNACK TINS DOILEY    set of 4 knick knack tins doiley            13
gift_0001_30  Dotcomgiftshop Gift Voucher £30.00   dotcomgiftshop gift voucher 30 00            13
      90184B AMETHYST CHUNKY BEAD BRACELET W STR amethyst chunky bead bracelet w str            13
       22016 Dotcomgiftshop Gift Voucher £100.00  dotcomgiftshop gift voucher 100 00            13
gift_0001_

In [35]:
max_sequence_length = int(
    np.percentile(token_lengths, 95)
)

print("95th percentile token length:",
    max_sequence_length)

max_sequence_length = max(
    max_sequence_length,
    16
)

print("Selected sequence length:",
    max_sequence_length)

batch_inputs = tokenizer(
    product_catalog["Normalized_Text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=max_sequence_length,
    return_tensors="pt"
)

print("\nInput IDs shape:",
    batch_inputs["input_ids"].shape)

print("Attention mask shape:",
    batch_inputs["attention_mask"].shape)

95th percentile token length: 10
Selected sequence length: 16

Input IDs shape: torch.Size([3957, 16])
Attention mask shape: torch.Size([3957, 16])


In [36]:
first_input_ids = batch_inputs["input_ids"][0]
first_attention_mask = batch_inputs["attention_mask"][0]

print("First input IDs:")
print(first_input_ids)

print("\nFirst attention mask:")
print(first_attention_mask)

print("\nActual tokens:")
print(
    tokenizer.convert_ids_to_tokens(
        first_input_ids[
            first_attention_mask == 1
        ].tolist()
    )
)

First input IDs:
tensor([  101,  1999, 10258, 27892,  2576,  7595,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0])

First attention mask:
tensor([1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0])

Actual tokens:
['[CLS]', 'in', '##fl', '##atable', 'political', 'globe', '[SEP]']


In [38]:
print("FINAL VALIDATION 3")
print("-" * 40)

print("Products:", len(product_catalog))
print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum token length in dataset:", token_lengths.max())
print("Median token length:", np.median(token_lengths))
print("95th percentile:", np.percentile(token_lengths, 95))
print("Selected sequence length:", max_sequence_length)
print("Products truncated:", truncated_count)
print("Batch input shape:", batch_inputs["input_ids"].shape)
print("Attention mask shape:", batch_inputs["attention_mask"].shape)

FINAL VALIDATION 3
----------------------------------------
Products: 3957
Tokenizer: distilbert-base-uncased
Vocabulary size: 30522
Maximum token length in dataset: 14
Median token length: 7.0
95th percentile: 10.0
Selected sequence length: 16
Products truncated: 0
Batch input shape: torch.Size([3957, 16])
Attention mask shape: torch.Size([3957, 16])


## Pretrained Transformer Embeddings

This section uses the pretrained DistilBERT model to generate dense semantic representations for product descriptions. The model is used without fine-tuning so that the resulting embeddings capture general language representations before being evaluated for product similarity.

In [39]:
import torch
from transformers import AutoModel

MODEL_NAME = "distilbert-base-uncased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print("Model loaded:", MODEL_NAME)

Device: cpu


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: distilbert-base-uncased


In [40]:
print("Hidden size:", model.config.hidden_size)
print("Number of layers:", model.config.n_layers)
print("Number of attention heads:", model.config.n_heads)

Hidden size: 768
Number of layers: 6
Number of attention heads: 12


In [41]:
with torch.no_grad():
    sample_inputs = {
        key: value.to(device)
        for key, value in batch_inputs.items()
    }

    sample_output = model(**sample_inputs)

print("Last hidden state shape:")
print(sample_output.last_hidden_state.shape)

Last hidden state shape:
torch.Size([3957, 16, 768])


In [42]:
first_product_hidden = sample_output.last_hidden_state[0]

print("First product hidden-state shape:")
print(first_product_hidden.shape)

print("\nFirst token representation:")
print(first_product_hidden[0])

print("\nRepresentation length:")
print(len(first_product_hidden[0]))

First product hidden-state shape:
torch.Size([16, 768])

First token representation:
tensor([-2.2260e-01, -1.5813e-01,  3.7652e-02, -4.3788e-02, -2.4417e-02,
         7.1454e-03, -1.2675e-02,  3.4506e-01, -2.3258e-01, -2.9877e-01,
         5.4349e-02, -1.0166e-01, -1.8849e-01,  3.6598e-01,  1.5616e-01,
         8.2797e-02, -2.9759e-01,  1.9147e-01,  1.2990e-01, -9.7967e-02,
        -9.8359e-02, -3.2805e-01, -1.4511e-01,  7.0916e-03, -4.0191e-02,
        -3.6398e-02, -1.1900e-01,  1.1215e-01,  9.5783e-02,  1.9343e-01,
         8.7430e-02,  8.1940e-02, -2.0936e-01, -1.2126e-01,  1.8678e-01,
        -1.5308e-01,  4.9329e-02, -7.1870e-02, -7.8054e-02,  6.3881e-02,
         5.2367e-02,  9.5391e-02,  3.4605e-01, -6.1203e-03, -1.2421e-01,
        -1.4129e-01, -2.2393e+00,  6.1627e-02, -3.3543e-01, -1.6877e-01,
         5.7045e-02, -1.1979e-01,  2.2473e-01,  2.3838e-01,  2.1443e-01,
         3.8211e-01,  9.6824e-02,  4.4854e-01,  1.4277e-01, -6.5019e-02,
         2.2921e-02,  2.9859e-02, -1.01

In [43]:
hidden_states = sample_output.last_hidden_state
attention_mask = batch_inputs["attention_mask"].to(device)

mask = attention_mask.unsqueeze(-1).expand(
    hidden_states.size()
).float()

masked_embeddings = hidden_states * mask

sum_embeddings = masked_embeddings.sum(dim=1)

sum_mask = mask.sum(dim=1).clamp(min=1e-9)

product_embeddings = sum_embeddings / sum_mask

print("Product embeddings shape:")
print(product_embeddings.shape)

Product embeddings shape:
torch.Size([3957, 768])


In [44]:
print("Embedding statistics:")
print(
    "Minimum:", product_embeddings.min().item()
)

print(
    "Maximum:", product_embeddings.max().item()
)

print(
    "Mean:", product_embeddings.mean().item()
)

print(
    "Standard deviation:",
    product_embeddings.std().item()
)

Embedding statistics:
Minimum: -4.291146755218506
Maximum: 1.2429652214050293
Mean: -0.01048513688147068
Standard deviation: 0.26108425855636597


In [46]:
print("NaN values:",
    torch.isnan(product_embeddings).sum().item())

print("Infinite values:",
    torch.isinf(product_embeddings).sum().item())

NaN values: 0
Infinite values: 0


In [47]:
import torch.nn.functional as F

normalized_embeddings = F.normalize(
    product_embeddings,
    p=2,
    dim=1
)

print("Normalized embedding shape:")
print(normalized_embeddings.shape)

print("\nFirst embedding norm:")
print(
    torch.norm(
        normalized_embeddings[0]
    ).item()
)

Normalized embedding shape:
torch.Size([3957, 768])

First embedding norm:
0.9999999403953552


In [49]:
product_embeddings_np = (
    normalized_embeddings
    .cpu()
    .numpy()
)

print("NumPy embedding shape:",
    product_embeddings_np.shape)

print("Data type:",
    product_embeddings_np.dtype)

NumPy embedding shape: (3957, 768)
Data type: float32


In [51]:
embedding_index = product_catalog[
    ["StockCode", "Product_Name", "Normalized_Text"]
].copy()

embedding_index["Embedding_Index"] = range(
    len(embedding_index)
)

print(embedding_index.head())

print("\nShape:",
    embedding_index.shape)

  StockCode                 Product_Name              Normalized_Text  \
0     10002   INFLATABLE POLITICAL GLOBE   inflatable political globe   
1     10080     GROOVY CACTUS INFLATABLE     groovy cactus inflatable   
2     10120                 DOGGY RUBBER                 doggy rubber   
3    10123C         HEARTS WRAPPING TAPE         hearts wrapping tape   
4    10124A  SPOTS ON RED BOOKCOVER TAPE  spots on red bookcover tape   

   Embedding_Index  
0                0  
1                1  
2                2  
3                3  
4                4  

Shape: (3957, 4)


In [52]:
import numpy as np

np.save(
    "../data/product_embeddings.npy",
    product_embeddings_np
)

embedding_index.to_csv(
    "../data/product_embedding_index.csv",
    index=False
)

print("Saved:")
print("../data/product_embeddings.npy")
print("../data/product_embedding_index.csv")

Saved:
../data/product_embeddings.npy
../data/product_embedding_index.csv


In [54]:
loaded_embeddings = np.load(
    "../data/product_embeddings.npy"
)

loaded_index = pd.read_csv(
    "../data/product_embedding_index.csv"
)

print("Loaded embeddings:",
    loaded_embeddings.shape)

print("Loaded index:",
    loaded_index.shape)

print(
    "\nEmbedding values match:",
    np.allclose(
        loaded_embeddings,
        product_embeddings_np
    )
)

Loaded embeddings: (3957, 768)
Loaded index: (3957, 4)

Embedding values match: True


In [55]:
print("FINAL VALIDATION 4")
print("-" * 40)

print("Products:", len(product_catalog))
print("Embedding dimension:", product_embeddings_np.shape[1])
print("Embedding matrix shape:", product_embeddings_np.shape)

print(
    "NaN values:",
    np.isnan(product_embeddings_np).sum()
)

print(
    "Infinite values:",
    np.isinf(product_embeddings_np).sum()
)

print(
    "First embedding norm:",
    np.linalg.norm(product_embeddings_np[0])
)

print(
    "Saved embedding file:",
    "../data/product_embeddings.npy"
)

print(
    "Saved index file:",
    "../data/product_embedding_index.csv"
)

FINAL VALIDATION 4
----------------------------------------
Products: 3957
Embedding dimension: 768
Embedding matrix shape: (3957, 768)
NaN values: 0
Infinite values: 0
First embedding norm: 0.99999994
Saved embedding file: ../data/product_embeddings.npy
Saved index file: ../data/product_embedding_index.csv


## Product Similarity and Semantic Search

This section uses the pretrained Transformer embeddings to identify semantically similar products. Cosine similarity is used to compare product representations and retrieve the most similar products for a given product query.

In [56]:
import numpy as np
import pandas as pd

product_embeddings = np.load(
    "../data/product_embeddings.npy"
)

embedding_index = pd.read_csv(
    "../data/product_embedding_index.csv"
)

print("Embeddings shape:", product_embeddings.shape)
print("Product index shape:", embedding_index.shape)

print("\nSample products:")
print(
    embedding_index[
        ["StockCode", "Product_Name", "Normalized_Text"]
    ].head()
)

Embeddings shape: (3957, 768)
Product index shape: (3957, 4)

Sample products:
  StockCode                 Product_Name              Normalized_Text
0     10002   INFLATABLE POLITICAL GLOBE   inflatable political globe
1     10080     GROOVY CACTUS INFLATABLE     groovy cactus inflatable
2     10120                 DOGGY RUBBER                 doggy rubber
3    10123C         HEARTS WRAPPING TAPE         hearts wrapping tape
4    10124A  SPOTS ON RED BOOKCOVER TAPE  spots on red bookcover tape


In [57]:
embedding_norms = np.linalg.norm(
    product_embeddings,
    axis=1
)

print("Minimum norm:", embedding_norms.min())
print("Maximum norm:", embedding_norms.max())
print("Mean norm:", embedding_norms.mean())

Minimum norm: 0.9999998
Maximum norm: 1.0000002
Mean norm: 1.0


In [58]:
def cosine_similarity_search(
    query_index,
    embeddings,
    index_df,
    top_k=10
):
    query_vector = embeddings[query_index]

    similarities = embeddings @ query_vector

    similarities[query_index] = -1

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    results = index_df.iloc[
        top_indices
    ][
        ["StockCode", "Product_Name", "Normalized_Text"]
    ].copy()

    results["Similarity"] = similarities[
        top_indices
    ]

    return results.reset_index(drop=True)

In [59]:
query_index = 0

query_product = embedding_index.iloc[
    query_index
]

print("QUERY PRODUCT")
print("-" * 40)
print("StockCode:", query_product["StockCode"])
print("Product:", query_product["Product_Name"])
print("Text:", query_product["Normalized_Text"])

print("\nSIMILAR PRODUCTS")
print("-" * 40)

similar_products = cosine_similarity_search(
    query_index=query_index,
    embeddings=product_embeddings,
    index_df=embedding_index,
    top_k=10
)

print(
    similar_products.to_string(index=False)
)

QUERY PRODUCT
----------------------------------------
StockCode: 10002
Product: INFLATABLE POLITICAL GLOBE
Text: inflatable political globe

SIMILAR PRODUCTS
----------------------------------------
StockCode                     Product_Name                  Normalized_Text  Similarity
    22389      PAPERWEIGHT SAVE THE PLANET      paperweight save the planet    0.825079
    10080         GROOVY CACTUS INFLATABLE         groovy cactus inflatable    0.814029
    22925    BLUE GIANT GARDEN THERMOMETER    blue giant garden thermometer    0.812501
    22927   GREEN GIANT GARDEN THERMOMETER   green giant garden thermometer    0.811781
    21369       MIRRORED WALL ART SPLODGES       mirrored wall art splodges    0.808890
    20682 RED RETROSPOT CHILDRENS UMBRELLA red retrospot childrens umbrella    0.807834
    21156        RETROSPOT CHILDRENS APRON        retrospot childrens apron    0.805616
    22926   IVORY GIANT GARDEN THERMOMETER   ivory giant garden thermometer    0.804652
    2122

In [60]:
search_term = "HEART"

matches = embedding_index[
    embedding_index["Product_Name"]
    .str.contains(
        search_term,
        case=False,
        na=False
    )
]

print(
    "Products matching:",
    search_term
)

print(
    matches[
        ["StockCode", "Product_Name"]
    ].head(20).to_string(index=False)
)

Products matching: HEART
StockCode                       Product_Name
   10123C               HEARTS WRAPPING TAPE
   16206B          RED PURSE WITH PINK HEART
   16207B             PINK HEART RED HANDBAG
    20669              RED HEART LUGGAGE TAG
    20707       CRAZY DAISY HEART DECORATION
    20832   RED FLOCK LOVE HEART PHOTO FRAME
    20845  ZINC HEART LATTICE 2 WALL PLANTER
    20846  ZINC HEART LATTICE T-LIGHT HOLDER
    20847   ZINC HEART LATTICE CHARGER LARGE
    20848   ZINC HEART LATTICE CHARGER SMALL
    20851       ZINC HEART LATTICE TRAY OVAL
    20854        BLUE PATCH PURSE PINK HEART
    20985                   HEART CALCULATOR
    20992         JAZZ HEARTS PURSE NOTEBOOK
    20996           JAZZ HEARTS ADDRESS BOOK
    21063          PARTY INVITES JAZZ HEARTS
    21114      LAVENDER SCENTED FABRIC HEART
    21143     ANTIQUE GLASS HEART DECORATION
    21188 3D HEARTS  HONEYCOMB PAPER GARLAND
    21190          PINK HEARTS PAPER GARLAND


In [61]:
if len(matches) > 0:

    selected_stockcode = matches.iloc[0]["StockCode"]

    query_index = embedding_index.index[
        embedding_index["StockCode"] == selected_stockcode
    ][0]

    print("QUERY PRODUCT")
    print("-" * 40)
    print(
        embedding_index.loc[
            query_index,
            "Product_Name"
        ]
    )

    print("\nSIMILAR PRODUCTS")
    print("-" * 40)

    similar_products = cosine_similarity_search(
        query_index=query_index,
        embeddings=product_embeddings,
        index_df=embedding_index,
        top_k=10
    )

    print(
        similar_products.to_string(index=False)
    )
else:
    print("No matching products found.")

QUERY PRODUCT
----------------------------------------
HEARTS WRAPPING TAPE

SIMILAR PRODUCTS
----------------------------------------
StockCode               Product_Name            Normalized_Text  Similarity
    21879           HEARTS GIFT TAPE           hearts gift tape    0.962990
    21883            STARS GIFT TAPE            stars gift tape    0.926824
    21882                SKULLS TAPE                skulls tape    0.902002
    22862      LOVE HEART NAPKIN BOX      love heart napkin box    0.892725
    22092    BLUE PAISLEY TISSUE BOX    blue paisley tissue box    0.885517
    20992 JAZZ HEARTS PURSE NOTEBOOK jazz hearts purse notebook    0.878779
    21881             CUTE CATS TAPE             cute cats tape    0.876304
    85195       HANGING HEART BASKET       hanging heart basket    0.875564
    21190  PINK HEARTS PAPER GARLAND  pink hearts paper garland    0.875115
   16156L             WRAP, CAROUSEL              wrap carousel    0.873186


In [62]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = AutoModel.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

print("Device:", device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cpu


In [63]:
def create_query_embedding(text):
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=16,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"]

    mask = attention_mask.unsqueeze(-1).expand(
        hidden_states.size()
    ).float()

    masked_embeddings = hidden_states * mask

    embedding = (
        masked_embeddings.sum(dim=1)
        / mask.sum(dim=1).clamp(min=1e-9)
    )

    embedding = F.normalize(
        embedding,
        p=2,
        dim=1
    )

    return embedding.cpu().numpy()[0]

In [64]:
def semantic_search(
    query,
    embeddings,
    index_df,
    top_k=10
):
    query_embedding = create_query_embedding(
        query
    )

    similarities = (
        embeddings @ query_embedding
    )

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    results = index_df.iloc[
        top_indices
    ][
        ["StockCode", "Product_Name", "Normalized_Text"]
    ].copy()

    results["Similarity"] = similarities[
        top_indices
    ]

    return results.reset_index(drop=True)

In [65]:
query = "red heart shaped decoration"

results = semantic_search(
    query=query,
    embeddings=product_embeddings,
    index_df=embedding_index,
    top_k=10
)

print("QUERY:")
print(query)

print("\nTOP RESULTS:")
print(
    results.to_string(index=False)
)

QUERY:
red heart shaped decoration

TOP RESULTS:
StockCode                        Product_Name                     Normalized_Text  Similarity
    22156        HEART DECORATION WITH PEARLS        heart decoration with pearls    0.924668
   85061W     WHITE JEWELLED HEART DECORATION     white jewelled heart decoration    0.917655
    23212   HEART WREATH DECORATION WITH BELL   heart wreath decoration with bell    0.910214
    22338        STAR DECORATION PAINTED ZINC        star decoration painted zinc    0.908910
    22335       HEART DECORATION PAINTED ZINC       heart decoration painted zinc    0.905171
    21385     IVORY HANGING DECORATION  HEART      ivory hanging decoration heart    0.900140
    22318       FIVE HEART HANGING DECORATION       five heart hanging decoration    0.897746
    23102      SILVER HEARTS TABLE DECORATION      silver hearts table decoration    0.895712
    21824 PAINTED METAL STAR WITH HOLLY BELLS painted metal star with holly bells    0.893746
    23213  

In [66]:
queries = [
    "red gift decoration",
    "kitchen storage container",
    "Christmas decoration",
    "children's toy",
    "hanging light decoration"
]

for query in queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    results = semantic_search(
        query=query,
        embeddings=product_embeddings,
        index_df=embedding_index,
        top_k=5
    )

    print(
        results[
            [
                "Product_Name",
                "Similarity"
            ]
        ].to_string(index=False)
    )


QUERY: red gift decoration
                    Product_Name  Similarity
  SILVER HEARTS TABLE DECORATION    0.925495
   SILVER BELLS TABLE DECORATION    0.911956
 WHITE CHRISTMAS STAR DECORATION    0.905608
          STAR DECORATION RUSTIC    0.905249
STAR WOODEN CHRISTMAS DECORATION    0.895606

QUERY: kitchen storage container
                    Product_Name  Similarity
   LARGE CERAMIC TOP STORAGE JAR    0.889790
   SMALL CERAMIC TOP STORAGE JAR    0.888900
VINTAGE CREAM CAT FOOD CONTAINER    0.864960
            LARGE POPCORN HOLDER    0.859528
            SMALL POPCORN HOLDER    0.859181

QUERY: Christmas decoration
                     Product_Name  Similarity
    TRADITIONAL CHRISTMAS RIBBONS    0.915806
    TRADITIONAL CHRISTMAS RIBBONS    0.915806
  PARTY CONE CHRISTMAS DECORATION    0.915502
 STAR WOODEN CHRISTMAS DECORATION    0.914943
HEART WOODEN CHRISTMAS DECORATION    0.909492

QUERY: children's toy
                 Product_Name  Similarity
          FLORAL SOFT CAR TO

In [68]:
query = "Christmas decoration"

query_embedding = create_query_embedding(
    query
)

similarities = (
    product_embeddings @ query_embedding
)

print("Similarity statistics:")
print(
    pd.Series(similarities).describe()
)

print("\nHighest similarity:",
    similarities.max())

print("Lowest similarity:",
    similarities.min())

Similarity statistics:
count    3957.000000
mean        0.683596
std         0.060457
min         0.489576
25%         0.643367
50%         0.681789
75%         0.721531
max         0.915806
dtype: float64

Highest similarity: 0.9158058
Lowest similarity: 0.48957568


In [69]:
query = "red heart shaped decoration"

results = semantic_search(
    query=query,
    embeddings=product_embeddings,
    index_df=embedding_index,
    top_k=10
)

results.to_csv(
    "../data/semantic_search_example.csv",
    index=False
)

print(
    "Saved:",
    "../data/semantic_search_example.csv"
)

Saved: ../data/semantic_search_example.csv


In [71]:
print("FINAL VALIDATION 5")
print("-" * 40)

print("Product embeddings:", product_embeddings.shape)
print("Product index:", embedding_index.shape)

print(
    "Embedding dimension:",
    product_embeddings.shape[1]
)

print(
    "Number of products:",
    len(embedding_index)
)

print(
    "Example search results:",
    len(results)
)

print(
    "\nTop semantic-search result:"
)

print(
    results.iloc[0][
        [
            "StockCode",
            "Product_Name",
            "Similarity"
        ]
    ]
)

FINAL VALIDATION 5
----------------------------------------
Product embeddings: (3957, 768)
Product index: (3957, 4)
Embedding dimension: 768
Number of products: 3957
Example search results: 10

Top semantic-search result:
StockCode                              22156
Product_Name    HEART DECORATION WITH PEARLS
Similarity                          0.924668
Name: 0, dtype: object


## Text Classification and Product Understanding

The original dataset does not provide product-category labels. Therefore, transparent keyword-based rules are used to create weak product categories. These labels are then used to evaluate whether pretrained Transformer embeddings can support product-category classification.

The resulting classification performance is treated as a weakly supervised experiment rather than ground-truth product classification.

In [72]:
import numpy as np
import pandas as pd

product_catalog = pd.read_csv(
    "../data/product_nlp_catalog.csv"
)

product_embeddings = np.load(
    "../data/product_embeddings.npy"
)

print("Product catalog:", product_catalog.shape)
print("Embeddings:", product_embeddings.shape)

Product catalog: (3957, 11)
Embeddings: (3957, 768)


In [88]:
import re

def contains_keyword(text, keywords):
    text = str(text).lower()

    for keyword in keywords:
        pattern = r"\b" + re.escape(keyword) + r"\b"

        if re.search(pattern, text):
            return True

    return False


def assign_product_category(text):
    text = str(text).lower()

    # Toys first because children's products can overlap
    if contains_keyword(text, [
        "toy",
        "game",
        "puzzle",
        "children",
        "childrens",
        "kids"
    ]):
        return "Toys"

    elif contains_keyword(text, [
        "candle",
        "t-light",
        "light holder",
        "lantern",
        "lamp"
    ]):
        return "Lighting"

    elif contains_keyword(text, [
        "mug",
        "cup",
        "plate",
        "bowl",
        "glass",
        "jug",
        "jar",
        "container",
        "storage"
    ]):
        return "Kitchenware"

    elif contains_keyword(text, [
        "bag",
        "purse",
        "wallet"
    ]):
        return "Bags"

    elif contains_keyword(text, [
        "christmas",
        "wreath",
        "decoration",
        "ornament",
        "garland",
        "tinsel"
    ]):
        return "Decorations"

    elif contains_keyword(text, [
        "heart",
        "love",
        "valentine"
    ]):
        return "Gifts"

    elif contains_keyword(text, [
        "tape",
        "paper",
        "notebook",
        "notepad",
        "pen",
        "pencil",
        "card",
        "book"
    ]):
        return "Stationery"

    elif contains_keyword(text, [
        "mirror",
        "clock",
        "frame",
        "wall",
        "ornamental"
    ]):
        return "Home Decor"

    elif contains_keyword(text, [
        "flower",
        "plant",
        "garden",
        "bird",
        "butterfly"
    ]):
        return "Garden"

    else:
        return "Other"

In [89]:
product_catalog["Category"] = (
    product_catalog["Normalized_Text"]
    .apply(assign_product_category)
)

print("Category distribution:")
print(
    product_catalog["Category"]
    .value_counts()
)

print("\nCategory percentages:")
print(
    (
        product_catalog["Category"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

Category distribution:
Category
Other          2153
Kitchenware     357
Decorations     262
Stationery      215
Lighting        208
Garden          183
Gifts           180
Bags            173
Home Decor      168
Toys             58
Name: count, dtype: int64

Category percentages:
Category
Other          54.41
Kitchenware     9.02
Decorations     6.62
Stationery      5.43
Lighting        5.26
Garden          4.62
Gifts           4.55
Bags            4.37
Home Decor      4.25
Toys            1.47
Name: proportion, dtype: float64


In [90]:
for category in product_catalog["Category"].unique():

    print("\n" + "=" * 60)
    print("CATEGORY:", category)
    print("=" * 60)

    examples = product_catalog[
        product_catalog["Category"] == category
    ][
        ["Product_Name", "Normalized_Text"]
    ].head(5)

    print(
        examples.to_string(index=False)
    )


CATEGORY: Other
                Product_Name              Normalized_Text
  INFLATABLE POLITICAL GLOBE   inflatable political globe
    GROOVY CACTUS INFLATABLE     groovy cactus inflatable
                DOGGY RUBBER                 doggy rubber
     MINI FUNKY DESIGN TAPES      mini funky design tapes
COLOURING PENCILS BROWN TUBE colouring pencils brown tube

CATEGORY: Stationery
               Product_Name             Normalized_Text
       HEARTS WRAPPING TAPE        hearts wrapping tape
SPOTS ON RED BOOKCOVER TAPE spots on red bookcover tape
   ARMY CAMO BOOKCOVER TAPE    army camo bookcover tape
ASSTD DESIGN RACING CAR PEN asstd design racing car pen
 PAPER POCKET TRAVELING FAN  paper pocket traveling fan

CATEGORY: Home Decor
         Product_Name       Normalized_Text
      FAN BLACK FRAME       fan black frame
SILVER LOOKING MIRROR silver looking mirror
GOLDIE LOOKING MIRROR goldie looking mirror
   SILVER PHOTO FRAME    silver photo frame
     GOLD PHOTO FRAME      gold pho

In [91]:
category_counts = (
    product_catalog["Category"]
    .value_counts()
)

valid_categories = category_counts[
    category_counts >= 30
].index

classification_df = product_catalog[
    product_catalog["Category"]
    .isin(valid_categories)
].copy()

print("Categories retained:")
print(
    classification_df["Category"]
    .value_counts()
)

print(
    "\nClassification dataset shape:",
    classification_df.shape
)

Categories retained:
Category
Other          2153
Kitchenware     357
Decorations     262
Stationery      215
Lighting        208
Garden          183
Gifts           180
Bags            173
Home Decor      168
Toys             58
Name: count, dtype: int64

Classification dataset shape: (3957, 12)


In [92]:
embedding_lookup = {
    stockcode: i
    for i, stockcode
    in enumerate(product_catalog["StockCode"])
}

classification_indices = [
    embedding_lookup[stockcode]
    for stockcode in classification_df["StockCode"]
]

X_embeddings = product_embeddings[
    classification_indices
]

y_categories = classification_df[
    "Category"
].values

print("X shape:", X_embeddings.shape)
print("y shape:", y_categories.shape)

X shape: (3957, 768)
y shape: (3957,)


In [93]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train_text, y_test_text = (
    train_test_split(
        X_embeddings,
        y_categories,
        test_size=0.20,
        random_state=42,
        stratify=y_categories
    )
)

print("Training shape:", X_train_text.shape)
print("Test shape:", X_test_text.shape)

print("\nTraining categories:")
print(
    pd.Series(y_train_text)
    .value_counts()
)

Training shape: (3165, 768)
Test shape: (792, 768)

Training categories:
Other          1722
Kitchenware     286
Decorations     210
Stationery      172
Lighting        166
Garden          146
Gifts           144
Bags            138
Home Decor      134
Toys             47
Name: count, dtype: int64


In [94]:
from sklearn.linear_model import LogisticRegression

text_classifier = LogisticRegression(
    max_iter=2000,
    random_state=42
)

text_classifier.fit(
    X_train_text,
    y_train_text
)

print("Classifier trained successfully.")

Classifier trained successfully.


In [95]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

y_pred_text = text_classifier.predict(
    X_test_text
)

accuracy = accuracy_score(
    y_test_text,
    y_pred_text
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_test_text,
        y_pred_text,
        average="weighted",
        zero_division=0
    )
)

print("Accuracy:", round(accuracy, 4))
print("Weighted Precision:", round(precision, 4))
print("Weighted Recall:", round(recall, 4))
print("Weighted F1:", round(f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test_text,
        y_pred_text,
        zero_division=0
    )
)

Accuracy: 0.6932
Weighted Precision: 0.7369
Weighted Recall: 0.6932
Weighted F1: 0.6426

Classification Report:
              precision    recall  f1-score   support

        Bags       0.90      0.26      0.40        35
 Decorations       0.80      0.71      0.76        52
      Garden       0.67      0.16      0.26        37
       Gifts       0.77      0.47      0.59        36
  Home Decor       0.91      0.29      0.44        34
 Kitchenware       0.83      0.28      0.42        71
    Lighting       0.91      0.48      0.62        42
       Other       0.66      0.98      0.79       431
  Stationery       0.78      0.16      0.27        43
        Toys       1.00      0.09      0.17        11

    accuracy                           0.69       792
   macro avg       0.82      0.39      0.47       792
weighted avg       0.74      0.69      0.64       792



In [96]:
labels = sorted(
    np.unique(y_test_text)
)

cm = confusion_matrix(
    y_test_text,
    y_pred_text,
    labels=labels
)

confusion_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print("Confusion Matrix:")
print(confusion_df)

Confusion Matrix:
             Bags  Decorations  Garden  Gifts  Home Decor  Kitchenware  \
Bags            9            2       0      1           0            0   
Decorations     0           37       0      0           0            0   
Garden          0            0       6      0           0            0   
Gifts           1            1       0     17           1            2   
Home Decor      0            0       0      0          10            0   
Kitchenware     0            1       2      4           0           20   
Lighting        0            0       0      0           0            0   
Other           0            4       1      0           0            2   
Stationery      0            1       0      0           0            0   
Toys            0            0       0      0           0            0   

             Lighting  Other  Stationery  Toys  
Bags                0     23           0     0  
Decorations         1     14           0     0  
Garden              

In [97]:
prediction_results = classification_df[
    [
        "StockCode",
        "Product_Name",
        "Category"
    ]
].copy()

prediction_results["Predicted_Category"] = (
    text_classifier.predict(
        product_embeddings[
            [
                embedding_lookup[x]
                for x in prediction_results["StockCode"]
            ]
        ]
    )
)

prediction_results["Correct"] = (
    prediction_results["Category"]
    ==
    prediction_results["Predicted_Category"]
)

print(
    prediction_results.head(20)
    .to_string(index=False)
)

StockCode                 Product_Name   Category Predicted_Category  Correct
    10002   INFLATABLE POLITICAL GLOBE      Other              Other     True
    10080     GROOVY CACTUS INFLATABLE      Other              Other     True
    10120                 DOGGY RUBBER      Other              Other     True
   10123C         HEARTS WRAPPING TAPE Stationery              Other    False
   10124A  SPOTS ON RED BOOKCOVER TAPE Stationery         Stationery     True
   10124G     ARMY CAMO BOOKCOVER TAPE Stationery         Stationery     True
    10125      MINI FUNKY DESIGN TAPES      Other              Other     True
    10133 COLOURING PENCILS BROWN TUBE      Other              Other     True
    10135 COLOURING PENCILS BROWN TUBE      Other              Other     True
    11001  ASSTD DESIGN RACING CAR PEN Stationery              Other    False
    15030              FAN BLACK FRAME Home Decor              Other    False
    15034   PAPER POCKET TRAVELING FAN Stationery         Statio

In [98]:
classification_df.to_csv(
    "../data/product_classification_dataset.csv",
    index=False
)

prediction_results.to_csv(
    "../data/product_category_predictions.csv",
    index=False
)

print("Saved:")
print(
    "../data/product_classification_dataset.csv"
)
print(
    "../data/product_category_predictions.csv"
)

Saved:
../data/product_classification_dataset.csv
../data/product_category_predictions.csv


In [99]:
print("FINAL VALIDATION 6")
print("-" * 40)

print(
    "Classification dataset:",
    classification_df.shape
)

print(
    "Number of categories:",
    classification_df["Category"].nunique()
)

print(
    "Categories:",
    sorted(
        classification_df["Category"].unique()
    )
)

print(
    "Training samples:",
    len(X_train_text)
)

print(
    "Test samples:",
    len(X_test_text)
)

print(
    "Accuracy:",
    round(accuracy, 4)
)

print(
    "Weighted F1:",
    round(f1, 4)
)

print(
    "Prediction results:",
    prediction_results.shape
)

FINAL VALIDATION 6
----------------------------------------
Classification dataset: (3957, 12)
Number of categories: 10
Categories: ['Bags', 'Decorations', 'Garden', 'Gifts', 'Home Decor', 'Kitchenware', 'Lighting', 'Other', 'Stationery', 'Toys']
Training samples: 3165
Test samples: 792
Accuracy: 0.6932
Weighted F1: 0.6426
Prediction results: (3957, 5)


## Traditional NLP vs Transformer Representations

In [109]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

product_df = pd.read_csv("../data/product_nlp_catalog.csv")

transformer_embeddings = np.load("../data/product_embeddings.npy")
embedding_index = pd.read_csv("../data/product_embedding_index.csv")

print("Product dataset shape:", product_df.shape)
print("Transformer embeddings shape:", transformer_embeddings.shape)
print("Embedding index shape:", embedding_index.shape)

print("\nColumns:")
print(product_df.columns.tolist())

Product dataset shape: (3957, 11)
Transformer embeddings shape: (3957, 768)
Embedding index shape: (3957, 4)

Columns:
['StockCode', 'Product_Name', 'Total_Units_Sold', 'Total_Revenue', 'Order_Count', 'Customer_Count', 'Text_Length', 'Word_Count', 'Normalized_Text', 'Normalized_Text_Length', 'Normalized_Word_Count']


In [110]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=5000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    product_df["Normalized_Text"].fillna("")
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF matrix shape: (3957, 3428)
Vocabulary size: 3428


In [111]:
def tfidf_search(query, top_k=5):
    query_vector = tfidf_vectorizer.transform([query.lower()])
    
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = product_df.iloc[top_indices][
        ["StockCode", "Product_Name", "Normalized_Text"]
    ].copy()
    
    results["Similarity"] = scores[top_indices]
    
    return results.reset_index(drop=True)

In [112]:
def transformer_search(query, top_k=5):
    query_inputs = tokenizer(
        query,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=16
    )

    with torch.no_grad():
        query_output = model(**query_inputs)

    query_embedding = query_output.last_hidden_state.mean(
        dim=1
    ).numpy()

    query_embedding = query_embedding / np.linalg.norm(
        query_embedding,
        axis=1,
        keepdims=True
    )

    scores = np.dot(
        transformer_embeddings,
        query_embedding.T
    ).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = product_df.iloc[top_indices][
        ["StockCode", "Product_Name", "Normalized_Text"]
    ].copy()

    results["Similarity"] = scores[top_indices]

    return results.reset_index(drop=True)

In [113]:
query = "red heart shaped decoration"

print("QUERY:", query)

print("\n--- TF-IDF RESULTS ---")
display(tfidf_search(query, top_k=5))

print("\n--- TRANSFORMER RESULTS ---")
display(transformer_search(query, top_k=5))

QUERY: red heart shaped decoration

--- TF-IDF RESULTS ---


,StockCode,Product_Name,Normalized_Text,Similarity
0,84975,HEART SHAPED MIRROR,heart shaped mirror,0.559608
1,22165,"DIAMANTE HEART SHAPED WALL MIRROR,",diamante heart shaped wall mirror,0.507799
2,23129,HEART SHAPED HOLLY WREATH,heart shaped holly wreath,0.492939
3,22435,SET OF 9 HEART SHAPED BALLOONS,set of 9 heart shaped balloons,0.492412
4,22822,CREAM WALL PLANTER HEART SHAPED,cream wall planter heart shaped,0.477527



--- TRANSFORMER RESULTS ---


,StockCode,Product_Name,Normalized_Text,Similarity
0,22156,HEART DECORATION WITH PEARLS,heart decoration with pearls,0.924668
1,85061W,WHITE JEWELLED HEART DECORATION,white jewelled heart decoration,0.917655
2,23212,HEART WREATH DECORATION WITH BELL,heart wreath decoration with bell,0.910214
3,22338,STAR DECORATION PAINTED ZINC,star decoration painted zinc,0.908910
4,22335,HEART DECORATION PAINTED ZINC,heart decoration painted zinc,0.905171


In [114]:
queries = [
    "red heart shaped decoration",
    "kitchen storage container",
    "christmas decoration",
    "gift tape",
    "children toy"
]

for query in queries:
    print("=" * 80)
    print("QUERY:", query)

    print("\nTF-IDF:")
    display(
        tfidf_search(query, top_k=5)[
            ["Product_Name", "Similarity"]
        ]
    )

    print("\nTransformer:")
    display(
        transformer_search(query, top_k=5)[
            ["Product_Name", "Similarity"]
        ]
    )

QUERY: red heart shaped decoration

TF-IDF:


,Product_Name,Similarity
0,HEART SHAPED MIRROR,0.559608
1,"DIAMANTE HEART SHAPED WALL MIRROR,",0.507799
2,HEART SHAPED HOLLY WREATH,0.492939
3,SET OF 9 HEART SHAPED BALLOONS,0.492412
4,CREAM WALL PLANTER HEART SHAPED,0.477527



Transformer:


,Product_Name,Similarity
0,HEART DECORATION WITH PEARLS,0.924668
1,WHITE JEWELLED HEART DECORATION,0.917655
2,HEART WREATH DECORATION WITH BELL,0.910214
3,STAR DECORATION PAINTED ZINC,0.908910
4,HEART DECORATION PAINTED ZINC,0.905171


QUERY: kitchen storage container

TF-IDF:


...

,Product_Name,Similarity
0,STARS GIFT TAPE,0.927077
1,HEARTS GIFT TAPE,0.925950
2,SKULLS TAPE,0.910106
3,HEARTS WRAPPING TAPE,0.891044
4,EMPIRE GIFT WRAP,0.866794


QUERY: children toy

TF-IDF:


,Product_Name,Similarity
0,METAL DECORATION NAUGHTY CHILDREN,0.551631
1,WHEELBARROW FOR CHILDREN,0.549803
2,CHILDREN'S SPACEBOY MUG,0.511539
3,DOLLY MIXTURE CHILDREN'S UMBRELLA,0.483191
4,CHILDREN'S APRON DOLLY GIRL,0.387073



Transformer:


,Product_Name,Similarity
0,TOYBOX WRAP,0.869749
1,FLORAL SOFT CAR TOY,0.865623
2,METAL DECORATION NAUGHTY CHILDREN,0.861205
3,CUTE CATS TAPE,0.859419
4,MRS ROBOT SOFT TOY,0.854593


In [115]:
print("TF-IDF dimensions:")
print("Number of products:", tfidf_matrix.shape[0])
print("Number of features:", tfidf_matrix.shape[1])

print("\nTransformer dimensions:")
print("Number of products:", transformer_embeddings.shape[0])
print("Embedding dimensions:", transformer_embeddings.shape[1])

TF-IDF dimensions:
Number of products: 3957
Number of features: 3428

Transformer dimensions:
Number of products: 3957
Embedding dimensions: 768


In [116]:
def word_overlap(query, product_name):
    query_words = set(query.lower().split())
    product_words = set(str(product_name).lower().split())
    
    if len(query_words) == 0:
        return 0
    
    return len(query_words & product_words) / len(query_words)


comparison_rows = []

for query in queries:
    tfidf_results = tfidf_search(query, top_k=5)
    transformer_results = transformer_search(query, top_k=5)

    tfidf_overlap = tfidf_results["Product_Name"].apply(
        lambda x: word_overlap(query, x)
    ).mean()

    transformer_overlap = transformer_results["Product_Name"].apply(
        lambda x: word_overlap(query, x)
    ).mean()

    comparison_rows.append({
        "Query": query,
        "TFIDF_Avg_Word_Overlap": tfidf_overlap,
        "Transformer_Avg_Word_Overlap": transformer_overlap
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

,Query,TFIDF_Avg_Word_Overlap,Transformer_Avg_Word_Overlap
0,red heart shaped decoration,0.500000,0.45
1,kitchen storage container,0.333333,0.20
2,christmas decoration,1.000000,0.80
3,gift tape,0.800000,0.70
4,children toy,0.200000,0.30


In [117]:
comparison_summary = pd.DataFrame({
    "Representation": [
        "TF-IDF",
        "DistilBERT"
    ],
    "Type": [
        "Lexical",
        "Contextual / Transformer"
    ],
    "Dimensions": [
        tfidf_matrix.shape[1],
        transformer_embeddings.shape[1]
    ],
    "Main_Strength": [
        "Exact word and phrase matching",
        "Semantic similarity and contextual representation"
    ],
    "Main_Limitation": [
        "Weak understanding of meaning",
        "More computationally expensive"
    ]
})

display(comparison_summary)

,Representation,Type,Dimensions,Main_Strength,Main_Limitation
0,TF-IDF,Lexical,3428,Exact word and phrase matching,Weak understanding of meaning
1,DistilBERT,Contextual / Transformer,768,Semantic similarity and contextual representation,More computationally expensive


### Conclusion

TF-IDF provides a strong and interpretable lexical baseline for product search, while Transformer embeddings provide a richer semantic representation.

TF-IDF is useful when query words closely match product descriptions. Transformer embeddings can capture relationships between words and product descriptions even when the wording is not identical.

For CommerceIQ, Transformer embeddings will therefore serve as the foundation for later semantic retrieval, while TF-IDF provides an important traditional NLP baseline.

The comparison is qualitative rather than a ground-truth benchmark because the product catalog does not contain manually labeled search relevance data.

## Save Transformer-Derived Features

In [118]:
import pandas as pd
import numpy as np

product_df = pd.read_csv("../data/product_nlp_catalog.csv")
transformer_embeddings = np.load("../data/product_embeddings.npy")

print("Product dataset shape:", product_df.shape)
print("Transformer embeddings shape:", transformer_embeddings.shape)

assert len(product_df) == len(transformer_embeddings), \
    "Product rows and embeddings do not match."

print("Row alignment check: PASSED")

Product dataset shape: (3957, 11)
Transformer embeddings shape: (3957, 768)
Row alignment check: PASSED


In [119]:
embedding_columns = [
    f"Transformer_Dim_{i+1}"
    for i in range(transformer_embeddings.shape[1])
]

embedding_df = pd.DataFrame(
    transformer_embeddings,
    columns=embedding_columns
)

embedding_df.insert(
    0,
    "StockCode",
    product_df["StockCode"].values
)

embedding_df.insert(
    1,
    "Product_Name",
    product_df["Product_Name"].values
)

print("Embedding feature table shape:", embedding_df.shape)
display(embedding_df.head())

Embedding feature table shape: (3957, 770)


,StockCode,Product_Name,Transformer_Dim_1,Transformer_Dim_2,Transformer_Dim_3,Transformer_Dim_4,Transformer_Dim_5,Transformer_Dim_6,Transformer_Dim_7,Transformer_Dim_8,...,Transformer_Dim_759,Transformer_Dim_760,Transformer_Dim_761,Transformer_Dim_762,Transformer_Dim_763,Transformer_Dim_764,Transformer_Dim_765,Transformer_Dim_766,Transformer_Dim_767,Transformer_Dim_768
0,10002,INFLATABLE POLITICAL GLOBE,-0.014267,-0.030666,0.013812,0.018648,0.016327,-0.005404,-0.039208,0.041318,...,0.023092,0.014449,0.047555,-0.035045,0.023487,-0.051292,-0.012611,-0.022645,0.012145,0.031623
1,10080,GROOVY CACTUS INFLATABLE,-0.020303,-0.016094,0.005879,0.030734,0.026832,-0.019664,-0.015436,0.083033,...,-0.007894,-0.005273,0.000981,-0.021974,-0.005745,-0.030775,-0.022984,0.009058,0.010441,0.004876
2,10120,DOGGY RUBBER,0.025122,-0.003880,0.011801,0.010723,0.021009,-0.013021,0.029785,0.034531,...,-0.003864,0.032427,-0.007294,-0.031294,0.036081,-0.045486,-0.021231,0.029438,0.031171,0.021570
3,10123C,HEARTS WRAPPING TAPE,0.049346,-0.013695,0.007904,0.020789,0.008704,-0.018144,-0.001073,-0.025803,...,0.037292,-0.022739,-0.005633,-0.019197,0.016322,-0.001596,-0.005379,0.024233,-0.017631,-0.013677
4,10124A,SPOTS ON RED BOOKCOVER TAPE,0.009500,-0.037538,-0.008975,0.036196,0.045097,-0.021901,-0.035174,0.014917,...,0.013015,-0.006007,0.000853,-0.017832,-0.003100,0.014180,-0.024165,0.013003,0.018559,0.003431


In [120]:
print("Missing values:", embedding_df.isna().sum().sum())
print("Infinite values:", np.isinf(
    embedding_df[embedding_columns].values
).sum())

print(
    "Minimum embedding value:",
    embedding_df[embedding_columns].values.min()
)

print(
    "Maximum embedding value:",
    embedding_df[embedding_columns].values.max()
)

print(
    "Average embedding norm:",
    np.linalg.norm(
        embedding_df[embedding_columns].values,
        axis=1
    ).mean()
)

Missing values: 0
Infinite values: 0
Minimum embedding value: -0.6578849
Maximum embedding value: 0.13837725
Average embedding norm: 1.0


In [121]:
output_path = "../data/product_transformer_features.csv"

embedding_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", embedding_df.shape)

Saved: ../data/product_transformer_features.csv
Shape: (3957, 770)


In [122]:
reloaded_features = pd.read_csv(output_path)

print("Reloaded shape:", reloaded_features.shape)

print(
    "Shape matches:",
    reloaded_features.shape == embedding_df.shape
)

print(
    "Values match:",
    np.allclose(
        reloaded_features[embedding_columns].values,
        embedding_df[embedding_columns].values,
        atol=1e-6
    )
)

Reloaded shape: (3957, 770)
Shape matches: True
Values match: True


C:\Users\Sadiya Sajid\AppData\Local\Temp\ipykernel_75676\3932871990.py:1: DtypeWarning: Columns (0: StockCode) have mixed types. Specify dtype option on import or set low_memory=False.
  reloaded_features = pd.read_csv(output_path)


In [123]:
import os

files_to_check = [
    "../data/product_embeddings.npy",
    "../data/product_embedding_index.csv",
    "../data/product_transformer_features.csv"
]

for file_path in files_to_check:
    size_mb = os.path.getsize(file_path) / (1024 ** 2)
    print(f"{file_path} → {size_mb:.2f} MB")

../data/product_embeddings.npy → 11.59 MB
../data/product_embedding_index.csv → 0.26 MB
../data/product_transformer_features.csv → 36.23 MB


## Conclusions

Established the NLP and Transformer foundation for CommerceIQ.

Key outcomes:

- Built a product-level NLP catalog containing 3,957 products.
- Normalized product descriptions while preserving useful product attributes.
- Tokenized product descriptions using DistilBERT.
- Generated 768-dimensional Transformer embeddings for all products.
- Built semantic product search using cosine similarity.
- Compared Transformer retrieval with a traditional TF-IDF baseline.
- Performed a weakly supervised product categorization experiment using transparent keyword-generated labels.
- Saved reusable Transformer-derived product features for downstream stages.

The weakly supervised classification experiment achieved moderate overall performance but showed strong class imbalance and noisy category boundaries. It is therefore treated as an experiment rather than a production-ready classifier.

The stronger result from this is the semantic representation and retrieval pipeline, which provides the foundation for the embedding, vector database, RAG, and agent components developed in later stages.

Important limitation:

The DistilBERT embeddings used here are general-purpose Transformer representations rather than embeddings specifically optimized for sentence similarity. Later stages can improve retrieval using dedicated embedding models and vector databases.

In [124]:
import os
import numpy as np
import pandas as pd

product_catalog = pd.read_csv(
    "../data/product_nlp_catalog.csv"
)

embeddings = np.load(
    "../data/product_embeddings.npy"
)

transformer_features = pd.read_csv(
    "../data/product_transformer_features.csv",
    low_memory=False
)

classification_dataset = pd.read_csv(
    "../data/product_classification_dataset.csv",
    low_memory=False
)

classification_predictions = pd.read_csv(
    "../data/product_category_predictions.csv",
    low_memory=False
)

print("FINAL__VALIDATION")
print("=" * 50)

print("Product NLP catalog:", product_catalog.shape)
print("Transformer embeddings:", embeddings.shape)
print("Transformer feature table:", transformer_features.shape)
print("Classification dataset:", classification_dataset.shape)
print("Classification predictions:", classification_predictions.shape)

print("\nRequired files:")
files = [
    "../data/product_nlp_catalog.csv",
    "../data/product_embeddings.npy",
    "../data/product_embedding_index.csv",
    "../data/semantic_search_example.csv",
    "../data/product_classification_dataset.csv",
    "../data/product_category_predictions.csv",
    "../data/product_transformer_features.csv"
]

for file_path in files:
    print(f"{file_path}: {'EXISTS' if os.path.exists(file_path) else 'MISSING'}")

FINAL__VALIDATION
Product NLP catalog: (3957, 11)
Transformer embeddings: (3957, 768)
Transformer feature table: (3957, 770)
Classification dataset: (3957, 12)
Classification predictions: (3957, 5)

Required files:
../data/product_nlp_catalog.csv: EXISTS
../data/product_embeddings.npy: EXISTS
../data/product_embedding_index.csv: EXISTS
../data/semantic_search_example.csv: EXISTS
../data/product_classification_dataset.csv: EXISTS
../data/product_category_predictions.csv: EXISTS
../data/product_transformer_features.csv: EXISTS


In [125]:
assert product_catalog.shape[0] == 3957
assert embeddings.shape == (3957, 768)
assert transformer_features.shape == (3957, 770)

assert product_catalog["Normalized_Text"].notna().all()
assert np.isfinite(embeddings).all()

assert os.path.exists("../data/semantic_search_example.csv")
assert os.path.exists("../data/product_classification_dataset.csv")
assert os.path.exists("../data/product_category_predictions.csv")
assert os.path.exists("../data/product_transformer_features.csv")

print("All integrity checks: PASSED")

All integrity checks: PASSED
